# Introduction to Multi-Arm Bandits

In this session, we will explore the multi-armed bandit problem, a classic problem in decision-making.

Imagine you are in a casino with several slot machines (or “arms”), each with an unknown probability of giving a reward. Your goal is to maximize your total reward over a series of pulls. It may sound simple at first, but this is not the case! The challenging part is that you do not know which machine is the best at the start, so you have to balance exploring different machines to learn their payouts and exploiting the machine you think is best at any point in time.

## Imports and Constants

In [1]:
import numpy as np
import time
from utils import simulate_bandit

## The Bandits Game


In this exercise, you will play a simple version of the multi-armed bandit problem. You are given 3 slot machines (arms). Each machine pays out a reward between 0 and 1, but each has a different average payout. The exact payout distribution for each machine is hidden from you.

Your task is to maximize your total reward over 10 pulls.

*   On each pull, you will be asked to choose a machine by entering 0, 1, or 2.
*   The machine will return a reward (a number between 0 and 1).
*   After each pull, you will see:
    *   The reward you just earned
    *   The history of rewards for each machine so far
    *   The cumulative reward across all your pulls

At the end of the game, you will see a summary showing:

*   The total reward you earned overall
*   The total and average reward for each machine

**Your goal**: Try to figure out which machine is best, while still exploring enough to be confident in your choice.

In [ ]:
def user_choice(history):
    while True:
      try:
          choice = int(input("Choose an arm (0, 1, or 2): "))
          if choice in [0, 1, 2]:
              break
          else:
              print("Please enter 0, 1, or 2.")
      except ValueError:
          print("Invalid input. Please enter a number (0, 1, or 2).")

    return choice

simulate_bandit(user_choice, steps=15)

**Exercise Reflection**

1.   Which machine do you think was the best?
      *   Did one arm give you consistently higher rewards on average?
2.   How did you make your choices?
      *   Did you explore all three machines?
      *   Did you mostly stick with one after seeing a few good rewards?
3.   What would you do differently if you had 100 pulls instead of 10?
      *   Would you spend more time exploring first?
      *   When would you decide to settle on the “best” machine?
4.   Was the outcome surprising?
      *   Did randomness ever make a worse machine look better in the short term?



















## Naive Strategies

Now that you have experienced the bandit game interactively, let’s think about ways to maximize your total reward. In general, the challenge is to balance exploration and exploitation:

*   Exploration: Trying different arms to learn about their reward distributions.

*   Exploitation: Using the information you already have to pick the arm you think is best.

Even without sophisticated algorithms, there are some naive strategies we can try to see how well they perform. Two simple approaches are:

### Random Selection

The main idea of the strategy implemented below is to always pull an arm at random.

In [ ]:
def random_strategy(history):
    # Hint: You do not have to use history for this function, since our selection is totally random.

    ... # fill in this line using np.random.choice(...) to return a choise from a list of numbers representing the possible arms.

simulate_bandit(random_strategy)

If you use this strategy, you will explore all machines equally, but you won’t use any information from previous pulls. Sometimes, you might pick the best machine by chance, but often you will waste pulls on machines that are worse. Over many pulls, your total reward will reflect the average of all machines, without improving as you learn which machine is better.

### Max Selection

This strategy is based on choosing the arm that gave the highest reward on its last pull after exploring each arm's reward once.

In [11]:
def max_last_reward_strategy(history):
    """
    Selects the action that most recently yielded the highest reward.

    Parameters
    ----------
    history : dict[int, list[float]]
        A dictionary mapping action indices to their historical rewards.
        Example format: {0: [r_0, r_1, ...], 1: [...], 2: [...]}, where each
        list contains the sequence of rewards received from taking that action.
        
    Returns
    -------
    int
        The index of the action (key in `history`) whose most recent reward 
        value is maximal among all actions. If multiple actions share the same 
        last reward, one of them is returned (e.g., the first encountered).
    """

    # Check that there is not an arm not picked yet
    for arm in history:
        ... # Fill this line using an if statement check that return an arm if a condition is met

    ... # Create a list, that contains the last reward for each arm in history, using [-1] and taking advantage of history.values()
    
    # Return the arm that provides us with the maximum reward using np.argmax() to find the correct arm (index)
    # Do not forget to wrap your solution with int(), in order to retrieve an integer
    return ...  

In [ ]:
simulate_bandit(max_last_reward_strategy)

This time you are exploiting the most recent “lucky” pull, hoping it indicates a good machine. Early luck can make you keep choosing a machine that isn’t actually the best. This strategy doesn’t consider the average reward of each machine over multiple pulls, so it can easily get stuck on a suboptimal choice.

These naive strategies are simple to implement and provide a good starting point to see the effect of different decision rules. Now, we will explore slightly more sophisticated approaches that balance exploration and exploitation more effectively.

## Improved Strategies

### Greedy Selection

The greedy algorithm is a simple approach to the multi-armed bandit problem. It begins by pulling each arm once to gather some initial rewards. After this exploration phase, it always selects the arm with the highest average reward observed so far.

In [ ]:
def greedy_strategy(history):
    for arm in history:
        if len(history[arm]) == 0:
            return arm


    # Create the avg_rewards list with the mean value (use np.mean()) for each value in history when the arm's history len > 0.
    ... 

    return int(np.argmax(avg_rewards))

simulate_bandit(greedy_strategy)

The above algorithm is purely exploitative: it relies only on past experience to decide the next action. While this can work well if the initial rewards correctly identify the best arm, it can also fail if early outcomes are misleading, since the algorithm does not continue to explore other options once it commits.

### ε-Greedy Selection

The ε-greedy algorithm extends the greedy approach by introducing a small amount of randomness to encourage exploration. It also begins by pulling each arm once to collect initial rewards. After that, the algorithm follows this rule at each step:
*   With probability ε (a small number like 0.1), it chooses an arm at random — this is the exploration step.

*   With probability 1 − ε, it chooses the arm with the highest average reward observed so far — this is the exploitation step.

In [ ]:
def greedy_strategy(history, epsilon=0.1):
    for arm in history:
        if len(history[arm]) == 0:
            return arm

    avg_rewards = [np.mean(hist) if len(hist) > 0 else 0 for hist in history.values()]

    # We generate a value from 0 to 1. If that value is bigger than epsilon we proceed with the "best" arm, else we pick at random.
    prob = np.random.random()
    if prob > epsilon:
        return int(np.argmax(avg_rewards))
    else:
        return np.random.choice([0, 1, 2])

simulate_bandit(greedy_strategy)

By occasionally exploring, ε-greedy reduces the risk of getting stuck on a suboptimal arm due to unlucky early rewards. Over time, this balance between exploration and exploitation helps the algorithm find and exploit the best arm more reliably than pure greedy.

### Thompson Sampling

Thompson Sampling is a probabilistic algorithm for solving the multi-armed bandit problem. Instead of relying only on averages or fixed exploration rules, it maintains a probability distribution over how good each arm might be.

At each step, the algorithm samples a value from each arm’s distribution, selects the arm with the highest sample, observes the reward, and updates that arm’s distribution.

In [ ]:
def thompson_sampling(history, alpha_beta_store={"alpha_beta": None}):
    if alpha_beta_store["alpha_beta"] is None:
        alpha_beta_store["alpha_beta"] = {arm: [1, 1] for arm in history}

    alpha_beta = alpha_beta_store["alpha_beta"]

    samples = [np.random.beta(alpha, beta) for alpha, beta in alpha_beta.values()]
    chosen_arm = int(np.argmax(samples))

    if history[chosen_arm]:
        last_reward = history[chosen_arm][-1]
        if last_reward > 0.5:
            alpha_beta[chosen_arm][0] += 1
        else:
            alpha_beta[chosen_arm][1] += 1

    alpha_beta_store["alpha_beta"] = alpha_beta

    return chosen_arm

simulate_bandit(thompson_sampling)

This method naturally balances exploration and exploitation:

*   Arms with uncertain rewards are explored more often because their probability distributions are wide.

*   Arms that consistently perform well become more certain, and the algorithm tends to exploit them more frequently.

Over time, Thompson Sampling converges toward playing the optimal arm with high probability, while still occasionally exploring alternatives to avoid missing better options.